# Validation analysis

Run after exporting actual Drive artifacts with `scripts/export_results.py`. This notebook reads metrics only and needs no GPU. Test evaluation and inference comparisons remain pending.


In [ ]:
from pathlib import Path
import csv
import json

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
export = root / "results" / "exports"
assert export.is_dir(), "Export artifacts into results/exports first."


In [ ]:
baseline = json.loads((export / "baseline" / "metrics.json").read_text())
base_ppl = baseline["metrics"]["ppl"]
print(f"Baseline validation perplexity: {base_ppl:.6f}")
for path in sorted((export / "runs").glob("*/summary.json")):
    summary = json.loads(path.read_text())
    best = summary["best_trained_checkpoint"]
    print(path.parent.name, "steps:", summary["completed_steps"],
          "stop:", summary.get("stop_reason", "not recorded"),
          "best step:", best["step"], "ppl:", best["ppl"],
          "reduction %:", 100 * (base_ppl - best["ppl"]) / base_ppl)


In [ ]:
with (export / "validation_history.csv").open() as file:
    rows = list(csv.DictReader(file))
for row in rows:
    print(f"{row['run']:20s} step={int(row['step']):4d} "
          f"loss={float(row['token_loss']):.6f} ppl={float(row['ppl']):.6f}")


## Interpretation

Select checkpoints by validation token loss, not the final training loss. A plateau with cosine decay does not prove global convergence. Compare long rank-8 and rank-16 runs using the same data and schedule; record actual steps if early stopping differs. Keep the test set untouched until selection is finalized.
